# 🎵 Регресія: Що робить пісню популярною?

Spotify аналізує кожну пісню і вираховує десятки характеристик: наскільки вона «танцювальна», гучна, енергійна, позитивна. А потім присвоює їй **popularity** — число від 0 до 100.

**Наше завдання:** навчити модель передбачати popularity треку на основі його музичних характеристик.

> На відміну від класифікації (де відповідь — *клас*), регресія передбачає *число*. Наприклад, не «популярний / непопулярний», а конкретне значення — 73.4.

## Імпортуємо датасет

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Датасет: 114 000 треків зі Spotify
df = pd.read_csv("./dataset.csv")
print(f"Рядків: {len(df):,}   Колонок: {df.shape[1]}")
df.head()

In [ ]:
# Пошук пісні за назвою (часткове входження, без урахування регістру)
query = "The Weeknd"  # змініть на потрібне імя артиста
results = df[df["artists"].str.contains(query, case=False, na=False)]
print(f"Знайдено {len(results)} треків за запитом «{query}»:")
results[["track_name", "artists", "album_name", "popularity"]].head(20)

## Попередня обробка

In [ ]:
# Перевіряємо пропущені значення
print("Пропущені значення:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Видаляємо рядки з пропущеними значеннями
df = df.dropna()

# Видаляємо дублікати (один і той самий трек може бути в різних плейлистах)
df = df.drop_duplicates(subset='track_id')

# Конвертуємо explicit з True/False у 1/0
df['explicit'] = df['explicit'].astype(int)

# Залишаємо тільки треки з ненульовою popularity (нульова часто означає «невідомо»)
df = df[df['popularity'] > 0]

print(f"\nПісля очищення: {len(df):,} треків")
df.head()

## Опис колонок датасету

| Колонка | Опис |
|---|---|
| **popularity** | Цільова змінна. Від 0 до 100 — наскільки трек популярний зараз |
| **danceability** | Наскільки пісня підходить для танців (0.0 – 1.0) |
| **energy** | Інтенсивність і активність треку (0.0 – 1.0). Важкий метал ≈ 1.0, класика ≈ 0.1 |
| **loudness** | Гучність у децибелах (зазвичай від −60 до 0) |
| **speechiness** | Частка мови у треку. >0.66 — подкаст, <0.33 — музика |
| **acousticness** | Наскільки акустичний трек (0.0 – 1.0) |
| **instrumentalness** | Наскільки багато інструменталу в треці. Чим більше інструменталу - тим менше вокалу (0.0 – 1.0) |
| **liveness** | Ймовірність, що запис зроблено вживу (0.0 – 1.0) |
| **valence** | Позитивність/настрій треку. 1.0 = дуже радісний, 0.0 = сумний |
| **tempo** | Темп у BPM (ударів на хвилину) |
| **duration_ms** | Тривалість треку в мілісекундах |
| **key** | Тональність (0 = До, 1 = До#, ..., 11 = Сі) |
| **mode** | Лад: 1 = мажор, 0 = мінор |
| **explicit** | Чи містить ненормативну лексику (0/1) |
| **time_signature** | Розмір такту (кількість ударів на такт, зазвичай 3, 4 або 5) |
| **track_genre** | Жанр треку (категоріальна змінна) |

## Графіки: знайомимось з даними

In [ ]:
# Розподіл popularity
# Яка середня популярність?
plt.figure(figsize=(8, 5))
plt.hist(df['popularity'], bins=40, color='mediumpurple', edgecolor='white')
plt.title('Розподіл Popularity')
plt.xlabel('Popularity (0–100)')
plt.ylabel('Кількість треків')
plt.axvline(df['popularity'].mean(), color='red', linestyle='--',
            label=f"Середнє: {df['popularity'].mean():.1f}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Топ-10 жанрів за середньою Popularity
genre_pop = df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 5))
genre_pop.plot.barh(color='deepskyblue')
plt.title('Топ-10 жанрів за середньою Popularity')
plt.xlabel('Середня Popularity')
plt.gca().invert_yaxis()
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Popularity: explicit vs не-explicit
plt.figure(figsize=(8, 5))
df.groupby('explicit')['popularity'].mean().plot.bar(color=['steelblue', 'tomato'])
plt.title('Popularity: цензурний vs нецензурний контент')
plt.xticks([0, 1], ['Без ненормативної', 'З ненормативною'], rotation=0)
plt.ylabel('Середня Popularity')
plt.tight_layout()
plt.show()

In [ ]:
# Danceability vs Popularity
sample = df.sample(2000, random_state=42)
plt.figure(figsize=(8, 5))
plt.scatter(sample['danceability'], sample['popularity'],
            alpha=0.3, color='mediumseagreen', s=10)
plt.title('Danceability vs Popularity')
plt.xlabel('Danceability')
plt.ylabel('Popularity')
plt.tight_layout()
plt.show()

In [ ]:
# Instrumentalness vs Popularity
sample = df.sample(2000, random_state=42)
plt.figure(figsize=(8, 5))
plt.scatter(sample['instrumentalness'], sample['popularity'],
            alpha=0.3, color='orchid', s=10)
plt.title('Instrumentalness vs Popularity')
plt.xlabel('Instrumentalness')
plt.ylabel('Popularity')
plt.tight_layout()
plt.show()

In [ ]:
# Liveness vs Popularity
plt.figure(figsize=(8, 5))
plt.scatter(sample['liveness'], sample['popularity'],
            alpha=0.3, color='darkorange', s=10)
plt.title('Liveness vs Popularity')
plt.xlabel('Liveness')
plt.ylabel('Popularity')
plt.tight_layout()
plt.show()

In [ ]:
# Тривалість vs Popularity
duration_min = sample['duration_ms'] / 60000
plt.figure(figsize=(8, 5))
plt.scatter(duration_min, sample['popularity'],
            alpha=0.3, color='teal', s=10)
plt.title('Тривалість vs Popularity')
plt.xlabel('Тривалість (хвилини)')
plt.ylabel('Popularity')
plt.xlim(0, 10)
plt.tight_layout()
plt.show()

In [ ]:
# Середня Popularity за тональністю (Key)
key_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
key_pop = df.groupby('key')['popularity'].mean()
plt.figure(figsize=(8, 5))
plt.bar(key_names, key_pop.values, color='cornflowerblue', edgecolor='white')
plt.title('Середня Popularity за тональністю (Key)')
plt.xlabel('Тональність')
plt.ylabel('Середня Popularity')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Середня Popularity: мінор vs мажор
mode_pop = df.groupby('mode')['popularity'].mean()
plt.figure(figsize=(8, 5))
plt.bar(['Мінор', 'Мажор'], mode_pop.values, color=['salmon', 'mediumseagreen'], edgecolor='white')
plt.title('Середня Popularity: мінор vs мажор')
plt.ylabel('Середня Popularity')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Топ-20 пісень за Popularity та їхні жанри
top10 = df.nlargest(20, 'popularity')[['track_name', 'artists', 'popularity', 'track_genre']]
labels = top10['track_name'] + '\n' + top10['artists']

plt.figure(figsize=(10, 12))
bars = plt.barh(labels, top10['popularity'], color='mediumpurple', edgecolor='white')
plt.xlabel('Popularity')
plt.title('Топ-20 найпопулярніших пісень')
plt.gca().invert_yaxis()
plt.xlim(0, 105)

# Підписуємо жанр на кожному барі
for bar, genre in zip(bars, top10['track_genre']):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             genre, va='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

## 🔍 Детектив даних: які фічі важливі?

Перед тим як будувати модель, подивимось — **які характеристики пісні найбільше пов'язані з popularity?**

Кореляція — це число від −1 до 1:
- **+1** → чим більше X, тим більше popularity
- **−1** → чим більше X, тим менше popularity  
- **≈ 0** → зв'язку майже немає

### 🕵️ Ваше завдання: оберіть фічі для моделі

У наступній клітинці ви **самі обираєте**, які фічі використовувати. Потім порівняємо результати!

In [ ]:
# ======== Оберіть фічі (можна додавати/видаляти) ========
SELECTED_FEATURES = [
    'duration_ms',
    'explicit',
    'danceability',
    'energy',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'time_signature',
]
# =========================================================

print(f"Обрано {len(SELECTED_FEATURES)} фіч: {SELECTED_FEATURES}")

## Розділяємо дані на тренувальну та тестову вибірки

In [ ]:
from sklearn.model_selection import train_test_split

X = df[SELECTED_FEATURES]
y = df['popularity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Тренувальна вибірка: {X_train.shape[0]:,} треків")
print(f"Тестова вибірка:     {X_test.shape[0]:,} треків")

## Як зрозуміти, чи добре працює модель?

Уявіть: модель каже, що popularity пісні = 60, а насправді = 80. Модель помилилась на 20 пунктів.

Ми використовуємо **три числа**, щоб оцінити якість моделі на всіх тестових піснях одразу:

---

### MAE (Mean Absolute Error) — середня помилка

Найпростіша метрика. Беремо кожну пісню, дивимось на скільки модель помилилась (ігноруючи знак «+» чи «−»), і рахуємо середнє.

**Приклад:** якщо MAE = 12, це означає: *«в середньому модель промахується на 12 пунктів popularity»*.

Чим менше MAE — тим краще.

---

### RMSE (Root Mean Squared Error) — середня помилка, але суворіша

Працює як MAE, але **сильніше штрафує за великі промахи**.

Наприклад 2 моделі:
- Модель А помиляється на 5, 5, 5, 5 пунктів - стабільно неточна
- Модель Б помиляється на 1, 1, 1, 17 пунктів - один раз дуже сильно промахнулася

MAE у них однаковий (5), але RMSE у моделі Б буде вищим, бо RMSE «помічає» ту одну велику помилку.

**Правило:** RMSE завжди ≥ MAE. Якщо RMSE набагато більше за MAE — значить, модель іноді дуже сильно промахується.

---

### R² (R-квадрат) — наскільки модель краща за «тупе вгадування»

Уявіть найпростішу «модель»: на будь-яку пісню вона відповідає одне число — середнє popularity всіх пісень. Це як відповідати на всі питання тесту однаково.

R² показує, **наскільки наша модель краща за таке вгадування**:
- **R² = 0** → модель не краща за просте середнє (погано)
- **R² = 0.5** → модель пояснює 50% закономірностей у даних
- **R² = 1.0** → модель ідеально передбачає кожну пісню

---

> **Запитання для роздумів:**
> 1. Модель передбачила popularity = 60, а реальне = 80. Яка помилка (MAE) для цього одного прикладу?
> 2. Якщо R² = 0.25 — це хороша модель? Підказка: вона пояснює лише 25% закономірностей.

## Лінійна регресія

Найпростіша модель: будує пряму лінію через хмару точок.

Формула: `popularity = w₁·danceability + w₂·energy + ... + b`

Модель «шукає» коефіцієнти (w) так, щоб сума квадратів помилок була мінімальною.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Pipeline: нормалізуємо дані, потім тренуємо модель
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression()),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

print('=== Лінійна регресія ===')
print(f'MAE:  {mae_lr:.2f}')
print(f'RMSE: {rmse_lr:.2f}')
print(f'R²:   {r2_lr:.4f}')

In [ ]:
# Графік: реальні vs передбачені значення
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: факт vs передбачення
sample_idx = np.random.choice(len(y_test), 1000, replace=False)
y_test_arr = y_test.values
axes[0].scatter(y_test_arr[sample_idx], y_pred_lr[sample_idx], alpha=0.3, s=10, color='steelblue')
axes[0].plot([0, 100], [0, 100], 'r--', linewidth=1.5, label='Ідеальна модель')
axes[0].set_xlabel('Реальна Popularity')
axes[0].set_ylabel('Передбачена Popularity')
axes[0].set_title('Лінійна регресія: факт vs передбачення')
axes[0].legend()

# Коефіцієнти моделі (важливість фіч)
coefs = pd.Series(
    lr_pipeline.named_steps['lr'].coef_,
    index=SELECTED_FEATURES
).sort_values()
colors = ['tomato' if v < 0 else 'mediumseagreen' for v in coefs]
coefs.plot.barh(ax=axes[1], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Коефіцієнти лінійної регресії')
axes[1].set_xlabel('Вплив на Popularity')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### Як покращити лінійну регресію?

R² вийшов низький — модель майже не вловлює закономірності. Чому?

Лінійна регресія шукає **пряму лінію** між фічами і popularity. Але в реальності зв'язок може бути **кривим** — наприклад, і дуже короткі, і дуже довгі пісні непопулярні, а «середні» — найкращі. Пряма лінія це не зловить.

Спробуємо два підходи:
1. **Polynomial Features** — додаємо квадрати фіч та їхні комбінації, щоб модель могла будувати криві
2. **Lasso** — автоматично прибирає зайві фічі, залишаючи тільки корисні

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

# ======== Параметри (можна змінювати) ========
POLY_DEGREE = 2        # ступінь полінома: спробуйте 2, 3 (обережно — 3 повільно!)
RIDGE_ALPHA = 10.0     # сила регуляризації: спробуйте 0.1, 1.0, 10, 100
# =============================================

# Додаємо квадрати та комбінації фіч
# Наприклад, замість просто [danceability, energy] модель також бачить
# [danceability², energy², danceability × energy]

poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=RIDGE_ALPHA)),
])

poly_pipeline.fit(X_train, y_train)
y_pred_poly = poly_pipeline.predict(X_test)

print('=== Лінійна регресія + Polynomial Features ===')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_poly):.2f}')
print(f'RMSE: {root_mean_squared_error(y_test, y_pred_poly):.2f}')
print(f'R²:   {r2_score(y_test, y_pred_poly):.4f}')
print(f'\nБуло фіч: {X_train.shape[1]} → Стало: {poly_pipeline.named_steps["poly"].n_output_features_}')
print(f'\nПорівняйте з базовою лінійною регресією вище — R² покращився?')

In [ ]:
from sklearn.linear_model import Lasso

# ======== Параметри (можна змінювати) ========
LASSO_ALPHA = 0.1      # сила регуляризації: спробуйте 0.01, 0.1, 1.0, 10
#   маленький alpha → залишає більше фіч
#   великий alpha  → «вимикає» більше фіч
# =============================================

# Lasso автоматично «вимикає» фічі, які не допомагають
# Ставить коефіцієнт = 0 для зайвих фіч

lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=LASSO_ALPHA, random_state=42)),
])

lasso_pipeline.fit(X_train, y_train)
y_pred_lasso = lasso_pipeline.predict(X_test)

print('=== Лінійна регресія + Lasso ===')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_lasso):.2f}')
print(f'RMSE: {root_mean_squared_error(y_test, y_pred_lasso):.2f}')
print(f'R²:   {r2_score(y_test, y_pred_lasso):.4f}')

# Які фічі Lasso залишив?
coefs = pd.Series(lasso_pipeline.named_steps['lasso'].coef_, index=SELECTED_FEATURES)
print(f'\nФічі, які Lasso вважає корисними (коефіцієнт ≠ 0):')
print(coefs[coefs != 0].sort_values().round(3))
print(f'\nФічі, які Lasso "вимкнув" (коефіцієнт = 0):')
removed = coefs[coefs == 0].index.tolist()
print(removed if removed else 'жодної — всі фічі корисні')

## Decision Tree Regressor

Дерево рішень ділить дані на прямокутні «зони» і в кожній зоні передбачає середнє значення.

Головний параметр — **max_depth**: глибина дерева. Занадто глибоке = перенавчання (memorizes training data), занадто мілке = недонавчання.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# ======== Параметри (можна змінювати) ========
MAX_DEPTH_DT = 5          # глибина дерева: спробуйте 2, 5, 10, None
MIN_SAMPLES_LEAF_DT = 5  # мінімум зразків у листі: спробуйте 1, 5, 20
# =============================================

dt_pipeline = Pipeline([
    ('dt', DecisionTreeRegressor(
        max_depth=MAX_DEPTH_DT,
        min_samples_leaf=MIN_SAMPLES_LEAF_DT,
        random_state=42,
    )),
])

dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)

mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = root_mean_squared_error(y_test, y_pred_dt)
r2_dt   = r2_score(y_test, y_pred_dt)

print("=== Decision Tree Regressor ===")
print(f"MAE:  {mae_dt:.2f}")
print(f"RMSE: {rmse_dt:.2f}")
print(f"R²:   {r2_dt:.4f}")

In [ ]:
# Як глибина дерева впливає на якість?
depths = [1, 2, 3, 4, 5, 7, 10, 15, 20, None]
train_r2 = []
test_r2  = []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_r2.append(r2_score(y_train, m.predict(X_train)))
    test_r2.append(r2_score(y_test,  m.predict(X_test)))

x_labels = [str(d) if d is not None else 'None' for d in depths]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_labels, train_r2, 'o-', label='Train R²', color='steelblue')
ax.plot(x_labels, test_r2,  's-', label='Test R²',  color='tomato')
ax.set_title('Перенавчання: R² vs глибина дерева')
ax.set_xlabel('max_depth')
ax.set_ylabel('R²')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nЗапитання: При якій глибині Train R² >> Test R²? Що це означає?")

In [ ]:
from sklearn.tree import plot_tree

# Невелике дерево (max_depth=3) для наочності
small_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
small_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(small_tree,
          feature_names=X_train.columns.tolist(),
          filled=True,
          rounded=True,
          fontsize=9,
          ax=ax)
ax.set_title('Візуалізація дерева рішень (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

## KNN Regressor

Для нового треку знаходить K найближчих сусідів у тренувальних даних і повертає **середнє** їхнє popularity.

**Важливо:** KNN чутливий до масштабу даних — треба нормалізувати!

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# ======== Параметри (можна змінювати) ========
N_NEIGHBORS_KNN = 10          # кількість сусідів: спробуйте 1, 5, 20, 50
WEIGHTS_KNN     = 'distance'  # 'uniform' або 'distance'
# =============================================

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(
        n_neighbors=N_NEIGHBORS_KNN,
        weights=WEIGHTS_KNN,
    )),
])

knn_pipeline.fit(X_train, y_train)
y_pred_knn = knn_pipeline.predict(X_test)

mae_knn  = mean_absolute_error(y_test, y_pred_knn)
rmse_knn = root_mean_squared_error(y_test, y_pred_knn)
r2_knn   = r2_score(y_test, y_pred_knn)

print("=== KNN Regressor ===")
print(f"MAE:  {mae_knn:.2f}")
print(f"RMSE: {rmse_knn:.2f}")
print(f"R²:   {r2_knn:.4f}")

### Пошук найкращих параметрів: GridSearchCV для KNN

In [ ]:
from sklearn.model_selection import GridSearchCV

# ======== Параметри для перебору (можна змінювати) ========
param_grid_knn = {
    'knn__n_neighbors': [5, 10, 20, 50],
    'knn__weights':     ['uniform', 'distance'],
    'knn__metric':      ['euclidean', 'manhattan'],
}
scoring_knn = 'r2'   # 'r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'
# ==========================================================

grid_knn = GridSearchCV(
    knn_pipeline,
    param_grid_knn,
    cv=5,
    scoring=scoring_knn,
    n_jobs=-1,
)
grid_knn.fit(X_train, y_train)

best_params_knn = {k.replace('knn__', ''): v for k, v in grid_knn.best_params_.items()}
print(f"Найкращі параметри: {best_params_knn}")
print(f"Найкращий {scoring_knn} (CV): {grid_knn.best_score_:.4f}")

y_pred_knn_best = grid_knn.predict(X_test)
print(f"\nR²  на тесті: {r2_score(y_test, y_pred_knn_best):.4f}")
print(f"MAE на тесті: {mean_absolute_error(y_test, y_pred_knn_best):.2f}")
print(f"RMSE на тесті: {root_mean_squared_error(y_test, y_pred_knn_best):.2f}")

## Порівняння всіх моделей

Запускаємо GridSearchCV для всіх трьох алгоритмів і порівнюємо результати.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import pandas as pd

# ======== Scoring (можна змінювати) ========
scoring = 'r2'   # 'r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'
# ===========================================

models = {
    'Linear Regression (Ridge)': {
        'pipeline': Pipeline([('scaler', StandardScaler()), ('model', Ridge())]),
        'params': {
            'model__alpha': [0.01, 0.1, 1.0, 10, 100],
        },
    },
    'Decision Tree': {
        'pipeline': Pipeline([('model', DecisionTreeRegressor(random_state=42))]),
        'params': {
            'model__max_depth':        [3, 5, 7, 10],
            'model__min_samples_leaf': [5, 10, 20],
        },
    },
    'KNN': {
        'pipeline': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor())]),
        'params': {
            'model__n_neighbors': [5, 10, 20, 50],
            'model__weights':     ['uniform', 'distance'],
        },
    },
    'Random Forest': {
        'pipeline': Pipeline([('model', RandomForestRegressor(random_state=42))]),
        'params': {
            'model__n_estimators':     [100],
            'model__max_depth':        [10, 20],
            'model__min_samples_leaf': [5, 10],
        },
    },
    'Gradient Boosting': {
        'pipeline': Pipeline([('model', GradientBoostingRegressor(random_state=42))]),
        'params': {
            'model__n_estimators':  [100],
            'model__max_depth':     [3, 5],
            'model__learning_rate': [0.05, 0.1],
        },
    },
}

results = []

for name, config in models.items():
    print(f'Тренуємо {name}...')
    grid = GridSearchCV(
        config['pipeline'],
        config['params'],
        cv=3,
        scoring=scoring,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)

    best_params = {k.replace('model__', ''): v for k, v in grid.best_params_.items()}

    results.append({
        'Модель':            name,
        f'Найкращий {scoring} (CV)': round(grid.best_score_, 4),
        'R² (тест)':         round(r2_score(y_test, y_pred), 4),
        'MAE (тест)':        round(mean_absolute_error(y_test, y_pred), 2),
        'RMSE (тест)':       round(root_mean_squared_error(y_test, y_pred), 2),
        'Параметри':         str(best_params),
    })

print('\nГотово!')

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R² (тест)', ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1
results_df.index.name = 'Місце'

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
results_df

In [ ]:
# Візуалізація порівняння
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_names = results_df['Модель'].tolist()
r2_vals  = results_df['R² (тест)'].tolist()
mae_vals = results_df['MAE (тест)'].tolist()

palette = ['gold', 'silver', 'peru', 'steelblue', 'mediumseagreen', 'orchid']
colors = palette[:len(model_names)]

# R²
axes[0].barh(model_names, r2_vals, color=colors)
axes[0].set_title('R² (чим більше — тим краще)')
axes[0].set_xlabel('R²')
axes[0].set_xlim(0, max(r2_vals) * 1.15)
axes[0].grid(alpha=0.3, axis='x')
for i, v in enumerate(r2_vals):
    axes[0].text(v + 0.005, i, str(v), va='center')

# MAE
axes[1].barh(model_names, mae_vals, color=colors)
axes[1].set_title('MAE (чим менше — тим краще)')
axes[1].set_xlabel('MAE (пункти Popularity)')
axes[1].grid(alpha=0.3, axis='x')
for i, v in enumerate(mae_vals):
    axes[1].text(v + 0.1, i, str(v), va='center')

plt.tight_layout()
plt.show()

## 🏆 Підсумки та запитання для роздумів

1. **Яка модель перемогла?** Чи збіглось з вашим очікуванням?

2. **Детектив даних:** Поверніться до секції «Обираємо фічі». Видаліть 2–3 фічі з найнижчою кореляцією і запустіть порівняння знову. Якість покращилась чи погіршала?

3. **Обмеження моделі:** Чому popularity важко передбачити лише за музичними характеристиками? Які фактори ми **не врахували**?

4. **Реальне застосування:** Якщо б ви були продюсером і мали цю модель — як би ви її використали? Чи довіряли б її прогнозам?